In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Импорт датасета

In [4]:
df = pd.read_csv("../data/movies_1990_2026.csv")
df["title_clean"] = df["title"].str.lower().replace(",", "").str.strip()

In [5]:
df

,id,title,year,description,genres,rating,num_votes,adult,title_clean
0,10729,Quick Change,1990,"With the aid of his girlfriend, Phyllis Potter...","['Comedy', 'Crime']",6.691,411,False,quick change
1,769,GoodFellas,1990,"The true story of Henry Hill, a half-Irish, ha...","['Drama', 'Crime']",8.453,14282,False,goodfellas
2,60898,Erotic Ghost Story,1990,"After one thousand years, three vixens transfo...","['Fantasy', 'Drama', 'Horror']",6.170,94,False,erotic ghost story
3,114,Pretty Woman,1990,"Vivian is a carefree, streetwise diamond in th...","['Romance', 'Comedy']",7.448,8954,False,pretty woman
4,242,The Godfather Part III,1990,In the midst of trying to legitimize his busin...,"['Crime', 'Drama', 'Thriller']",7.400,6826,False,the godfather part iii
...,...,...,...,...,...,...,...,...,...
11095,1413097,Melania,2026,Offering unprecedented access to the 20 days l...,['Documentary'],3.600,150,False,melania
11096,1442669,La Vénus électrique,2026,Artist Antoine has lost all inspiration and sl...,"['Comedy', 'Drama', 'History']",0.000,0,False,la vénus électrique
11097,1400763,Touch Me,2026,Two codependent best friends become addicted t...,"['Horror', 'Science Fiction', 'Comedy']",6.000,8,False,touch me
11098,1196946,Ikkis,2026,Second Lieutenant and India's youngest Param V...,"['History', 'War', 'Drama']",6.000,13,False,ikkis


In [8]:
dict_df = dict(zip(df["title_clean"], df.index))


In [9]:
def random_films(query_titles, n = 10):
    
    candidates = df.copy()
    result = candidates.sample(n=min(n, len(candidates)))
    return [
        {
            "title": row["title"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "rating": round(float(row["rating"]), 1),
            "genres": row["genres"]
        }
        for _, row in result.iterrows()
    ]
    

In [10]:
random_films("The Matrix, Inception", n=5)

[{'title': 'Trapped',
  'year': 2002,
  'rating': 6.2,
  'genres': "['Thriller', 'Crime']"},
 {'title': 'Remember',
  'year': 2015,
  'rating': 7.4,
  'genres': "['Drama', 'Thriller', 'Mystery']"},
 {'title': 'The Legend of Gingko',
  'year': 2000,
  'rating': 5.4,
  'genres': "['Action', 'Adventure', 'Romance']"},
 {'title': 'The Rainmaker',
  'year': 1997,
  'rating': 7.0,
  'genres': "['Drama', 'Crime', 'Thriller']"},
 {'title': 'Rebirth of Mothra',
  'year': 1996,
  'rating': 6.2,
  'genres': "['Action', 'Adventure', 'Science Fiction', 'Fantasy', 'Family']"}]

In [11]:
def recommend_popular(query_titles, n = 10):
    candidates = df.copy()

    result = candidates.sort_values(by="rating", ascending=False).head(n)
    
    return [
        {
            "title": row["title"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "rating": round(float(row["rating"]), 1),
            "genres": row["genres"],
            "method": "popular"
        }
        for _, row in result.iterrows()
    ]

In [12]:
a = recommend_popular("Anora")
print(a)

[{'title': 'Yoon-Yool and Russian Sexy Woman', 'year': 2022, 'rating': 10.0, 'genres': "['Romance']", 'method': 'popular'}, {'title': "Yoon-Yool's Men Affairs", 'year': 2021, 'rating': 10.0, 'genres': "['Drama', 'Romance']", 'method': 'popular'}, {'title': 'The Pretty Ghostress Story', 'year': 1992, 'rating': 10.0, 'genres': "['Horror']", 'method': 'popular'}, {'title': 'Queen of Kowloon', 'year': 2000, 'rating': 10.0, 'genres': "['Drama']", 'method': 'popular'}, {'title': 'Salome', 'year': 2023, 'rating': 10.0, 'genres': "['Drama', 'Romance']", 'method': 'popular'}, {'title': 'Scissors', 'year': 2026, 'rating': 10.0, 'genres': "['Horror', 'Comedy']", 'method': 'popular'}, {'title': 'High K', 'year': 2000, 'rating': 10.0, 'genres': "['Crime']", 'method': 'popular'}, {'title': 'Bikini Bar: Delicious Service', 'year': 2020, 'rating': 10.0, 'genres': "['Romance', 'Drama']", 'method': 'popular'}, {'title': "God's Here", 'year': 2024, 'rating': 10.0, 'genres': "['Drama']", 'method': 'popula

In [9]:
def recommend_description(query_titles, n=10): 

    tfid = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
    df["description"] = df["description"].fillna(" ")
    description_matrix = tfid.fit_transform(df["description"])
    similarity_matrix = cosine_similarity(description_matrix).astype(np.float32)

    indices = []

    for title in query_titles.split(","):
        clean_title = title.lower().strip()
        if clean_title in dict_df:
            indices.append(dict_df[clean_title])

    if not indices:
        return "Error, don't have your films in database("

    seed_vectors = similarity_matrix[indices]

    avg_sim_scores = np.mean(seed_vectors, axis=0)

    for idx in indices:
        avg_sim_scores[idx] = -1

    top_indices = np.argsort(avg_sim_scores)[-n - 70 :]
    top_indices = top_indices[::-1]

    candidates = df.iloc[top_indices].copy()
    candidates["sim_score"] = avg_sim_scores[top_indices]

    candidates = candidates.sort_values(
       by=["sim_score", "rating"], ascending=[False, False]
        )

    result_df = candidates.head(n)

    recommendations = []

    for _, row in result_df.iterrows():
        recommendations.append(
            {
                "title": row["title"],
                "year": int(row["year"]) if pd.notna(row["year"]) else None,
                "rating": round(float(row["rating"]), 1),
                "genres": row["genres"],
                "similarity_score": round(float(row["sim_score"]), 4),
            }
        )
    return recommendations
    

In [14]:
a = recommend_description("Anoa", n=10) 
a

"Error, don't have your films in database("